# Selección, negocio, Registry y reproducción

Preparado sin salidas hasta que finalicen los controles, la selección y la reproducción. Los dos finalistas y los controles estáticos se congelan por CV antes de la evaluación final. 2014J fue observado en v1: se declara benchmark retrospectivo reutilizado, no prueba inédita. Consultar resultados guardados no es una nueva selección sobre prueba.


In [ ]:
from pathlib import Path
import os, sys, json
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
here = Path.cwd().resolve()
ROOT = next((p for p in [here, *here.parents] if (p/'config/protocolo-v2.json').is_file()), None)
if ROOT is None and (here/'pset-03/v2/config/protocolo-v2.json').is_file():
    ROOT = here/'pset-03/v2'
assert ROOT is not None, 'Abrir el notebook desde v2/notebooks o desde la raíz del repositorio'
os.environ['PSET3_V2_ROOT'] = str(ROOT)
sys.path.insert(0, str(ROOT))
from src.seguimiento import protocolo
from src.common import sha256
plan = protocolo()
def report(relative):
    path = ROOT/relative
    if not path.is_file():
        raise RuntimeError('Etapa pendiente: falta '+relative+'. Revisar docs/estado-v2.json y esperar al coordinador; no sustituir por resultados de v1.')
    return json.loads(path.read_text())
print('Protocolo verificado:', plan['id'])
print('Este notebook consulta resultados reales y reproduce sus cálculos; no inicia nuevas búsquedas ni llamadas de inferencia API.')


In [ ]:
decision=report('docs/negocio/finalistas.json')
from src.common import identity
assert decision['decision_id']==identity(decision['content'])
d=decision['content']
assert d['protocol_id']==plan['id'] and d['v2_holdout_evaluated'] is False
display(pd.DataFrame([{'rol':role,'familia':d[role]['family'],'run_id':d[role]['run_id'],'cobertura_CV':d[role]['cv_coverage_mean'],'DE_CV':d[role]['cv_coverage_std'],'AP_CV':d[role]['cv_ap_fail_mean']} for role in ['winner','challenger']]))
print('Decision ID:',decision['decision_id'])
print('Diferencia:',d['winner_challenger_difference'],'| variación:',d['fold_variation_reference'])
print('Supera variación entre folds:',d['difference_exceeds_fold_variation'])
display(pd.DataFrame(d['tuning_improvements']))


In [ ]:
final=report('docs/seleccion/evaluacion.json')
assert final['status']=='registered' and final['decision_id']==decision['decision_id']
assert final['winner_changed_using_test'] is False and final['test_previously_observed_in_v1'] is True
rows=[]
for name,r in final['models'].items():
    rows.append({'modelo':name,'familia':r['family'],**r['technical'],**r['business']['0.1'],
                 'inferencia_prueba_seg':r['test_inference_seconds'],'API_prueba_USD':r['test_api_cost_usd']})
display(pd.DataFrame(rows))
display(pd.DataFrame([final['longitudinal_vs_matched_static']]))
display(pd.DataFrame([final['reactive']]))
display(pd.DataFrame(final['registry']))
print('Run de evaluación final:',final['final_run_id'])


## Umbral, costo e interpretación

El cupo se aplica por edición: ordenar P(Fail) de mayor a menor y usar la clave solo para desempatar. No se impone un umbral universal de 0,5. Los cortes observados y los escenarios de costos relativos FN/FP se reportan; no se inventan precios institucionales. La regla busca máxima captura esperada con cupo fijo ocupado y costos homogéneos.

Dummy y reactiva son las dos baselines. El modelo estático, ajustado independientemente dentro de la misma familia, es el control que mide el aporte longitudinal. La reactiva asigna apoyo después del cierre, con capacidad acotada, y su cobertura preventiva es cero. Las diferencias predictivas no estiman el efecto causal de la tutoría.


In [ ]:
display(pd.read_csv(ROOT/'docs/seleccion/por_edicion.csv'))
display(pd.read_csv(ROOT/'docs/seleccion/costos.csv'))
display(pd.read_csv(ROOT/'docs/seleccion/explicabilidad.csv').sort_values('mean_coverage_drop',ascending=False).head(15))
display(pd.DataFrame([report('docs/seleccion/explicacion_local.json')]))


## Reproducción en proceso limpio

Se carga champion desde Model Registry y se recalcula AP y cobertura. Las nuevas peticiones API están bloqueadas durante la predicción; un LLM debe reproducir desde sus respuestas archivadas. Esta verificación no modifica la elección ni abre una nueva búsqueda sobre prueba.


In [ ]:
import subprocess
result=subprocess.run([sys.executable,str(ROOT/'scripts/reproducir.py')],cwd=ROOT,capture_output=True,text=True)
assert result.returncode==0, result.stdout+result.stderr
reproduced=report('docs/seleccion/reproduccion-champion.json')
assert reproduced['status']=='ok' and reproduced['fresh_process']
assert reproduced['ap_exact'] and reproduced['business_exact'] and reproduced['api_new_requests']==0
display(pd.DataFrame([reproduced]))
